# Bygge grafen

Målet er å skape en graf som
- Er fri for skala, og
- som også holder våre kriminelle.
Det vil i praksis si å lese inn grafen, og så legge til nodene (og deres relasjoner) slik vi har forberedt.

For at våre kriminelle skal gli naturlig inn, må de gis relasjoner til de eksisterende på en "naturlig" måte.  Det vil si at hver node må få (et antall) relasjoner tli andre noder som "ligner".  I praksis betyr det å ha større sjande for å knytte seg til noder som alerede har mange kanter.

Nå har (noen) av våre kriminelle trolig flere kanter enn medianen i datasettet, men det kan vi se på senere.

Eneste måten jeg har funnet for å legge inn ny noder på en "normal" måte, er å lage et sett av alle noder, hvor hver node er i settet like mange ganger som det har kanter.  Når én node nå trekkes fra settet vil det sannsynligheten for å trekke en node reflektere nodens sentralitet.





In [8]:
# Laste inn datasettet
# La oss lage en graf
import networkx as nx
import gzip

EG = nx.DiGraph()
# husk at gzip åpner i 'b'
with gzip.open("data/email.edgelist.txt.gz", "rt") as fd:
    for linje in fd:
         link = linje.split()
         EG.add_edge(int(link[0]), int(link[1]))
    #
#

# Merke nodene
nx.set_node_attributes(EG, True, name="Epost")

# Bort med eposter sendt til seg selv
EG.remove_edges_from(nx.selfloop_edges(EG))
isolerte = list(nx.isolates(EG)) # kan ikke bruke iteratorer direkte
EG.remove_nodes_from(isolerte)

print(f"Antall noder: {EG.number_of_nodes()}")
print(f"Andtall kanter: {EG.number_of_edges()}")


Antall noder: 57189
Andtall kanter: 103083


In [ ]:
# Hvor mange deler består grafen av.
# Vi bryr oss ikke om retningen
print(f"Grafen består av {nx.number_weakly_connected_components(EG)} deler")

# Grafen er rettet
weak_comp = list(nx.weakly_connected_components(EG))

# Summary using a dictionary: {Component_Index: Node_Count}
counts = {i: len(nodes) for i, nodes in enumerate(weak_comp, 1)}
print(counts)


185
{1: 56576, 2: 5, 3: 8, 4: 2, 5: 3, 6: 4, 7: 4, 8: 2, 9: 4, 10: 8, 11: 2, 12: 3, 13: 2, 14: 2, 15: 2, 16: 3, 17: 6, 18: 3, 19: 3, 20: 6, 21: 2, 22: 2, 23: 2, 24: 2, 25: 3, 26: 2, 27: 2, 28: 2, 29: 3, 30: 2, 31: 3, 32: 2, 33: 2, 34: 3, 35: 3, 36: 2, 37: 2, 38: 2, 39: 2, 40: 3, 41: 2, 42: 2, 43: 4, 44: 2, 45: 2, 46: 2, 47: 2, 48: 7, 49: 146, 50: 2, 51: 2, 52: 2, 53: 2, 54: 3, 55: 2, 56: 3, 57: 2, 58: 3, 59: 2, 60: 3, 61: 2, 62: 2, 63: 2, 64: 2, 65: 2, 66: 4, 67: 4, 68: 4, 69: 3, 70: 4, 71: 3, 72: 3, 73: 3, 74: 2, 75: 3, 76: 2, 77: 2, 78: 2, 79: 2, 80: 2, 81: 3, 82: 2, 83: 2, 84: 2, 85: 2, 86: 3, 87: 3, 88: 4, 89: 2, 90: 2, 91: 2, 92: 4, 93: 2, 94: 2, 95: 11, 96: 5, 97: 2, 98: 2, 99: 2, 100: 4, 101: 3, 102: 2, 103: 2, 104: 2, 105: 2, 106: 2, 107: 2, 108: 2, 109: 2, 110: 2, 111: 2, 112: 2, 113: 3, 114: 2, 115: 2, 116: 3, 117: 2, 118: 2, 119: 2, 120: 2, 121: 2, 122: 3, 123: 2, 124: 2, 125: 3, 126: 2, 127: 2, 128: 2, 129: 2, 130: 2, 131: 2, 132: 2, 133: 3, 134: 2, 135: 2, 136: 2, 137: 2, 

In [ ]:
# Finne settet av noder i den største
største = max(nx.weakly_connected_components(EG), key=len)
EPOST = EG.subgraph(largest_weak).copy()

In [9]:
UG = EG.to_undirected(EG)

print(f"Antall noder etter konvertering: {UG.number_of_nodes()}")
print(f"Andtall kanter etter konvertering: {UG.number_of_edges()}")

Antall noder etter konvertering: 57189
Andtall kanter etter konvertering: 92442


In [10]:
# Er det én graf
if nx.is_connected(UG):
    print("Grafen henger sammen")
else:
   # Hvor mange deler
   print(f"Består av {nx.number_connected_components(UG)} deler")
#

Består av 185 deler


Da bygger vi listen med noder, hvor antallet følger av antall kanter.  Det vil si at populære noder finnes ofte i listen.

In [16]:
alle_noder = []
for u, v in UG.edges():
    # En kant gir to noder.
    alle_noder.extend([u,v])
#
print(f"Antall noder i listen: {len(alle_noder)}")

Antall noder i listen: 184884


Jeg spør Gemini:
```
Using networkx, without converting the whole graph to a list, how can I find a nrandom node
```
Den svarer
```
random_node = random.sample(G.nodes, 1)[0]
```
Men når jeg fortsetter:
```
Are you sure?  In your code "random_node = random.sample(G.nodes, 1)[0]" seems to create a list before returning one element
```
Svaret er
```
To be intellectually honest: internally, it still performs an $O(n)$ operation, [...]
```
Eller, som alltid: Livet er lettere når man vet svaret.

In [19]:
# Laste inn bakmann
Bakmann = nx.read_graphml("grafer/Bakmann.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Bakmann, True, name="Bakmann")
# Sjekke at det ser bra ut  
print(f"Kanter: {Bakmann.number_of_edges()}")
print(f"Noder: {Bakmann.number_of_nodes()}")


Kanter: 49
Noder: 24


In [20]:
# Laste inn "money mule"
Esel = nx.read_graphml("grafer/Mule10.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Esel, True, name="Esel")
# Verify by checking the number of nodes and edges
print(f"Noder: {Esel.number_of_nodes()}")
print(f"Kanter: {Esel.number_of_edges()}")

Noder: 44
Kanter: 119


In [ ]:
# Laste inn deling av utbytte
Utbytte = nx.read_graphml("grafer/Utbytte.graphml")
# For å verifisere at vi finner de riktige nodene
nx.set_node_attributes(Utbytte, True, name="Utbytte")
# Verify by checking the number of nodes and edges
print(f"Noder: {Utbytte.number_of_nodes()}")
print(f"Kanter: {Utbytte.number_of_edges()}")

Noder: 81
Kanter: 159


In [23]:
import random

# Legg de små grafene til i den store
Komplett = UG # Bare for å skape variabelen
print(f"Noder: {Komplett.number_of_nodes()}")
print(f"Kanter: {Komplett.number_of_edges()}")

for g in Bakmann, Esel, Utbytte:
    Komplett = nx.disjoint_union(Komplett, g)
    for n in g:
        # Hent en tilfeldig vedktet node fra den originale grafen
        Komplett.add_edge(n, random.choice(alle_noder))
#
print(f"Kanter: {Komplett.number_of_nodes()}")
print(f"Edges: {Komplett.number_of_edges()}")

Noder: 57189
Kanter: 92442
Kanter: 57487
Edges: 92918


In [24]:
# Lagre grafen
nx.write_graphml(Komplett, "grafer/Komplett.graphml")